In [1]:
import os
import json
import shutil
from PIL import Image
from collections import defaultdict

In [2]:

DATASET_ROOT = r"../dataset/dataset_splits"
OUTPUT_ROOT = os.path.join(DATASET_ROOT, "../splitted_coco_dataset")
SPLITS = ["train", "val", "test"]
IMG_EXTS = [".jpg", ".jpeg", ".png"]

# unique class IDs collecting
all_class_ids = set()

for split in SPLITS:
    lbl_dir = os.path.join(DATASET_ROOT, split, "labels")
    for f in os.listdir(lbl_dir):
        if not f.endswith(".txt"):
            continue
        with open(os.path.join(lbl_dir, f)) as fh:
            for line in fh:
                cls = int(line.split()[0])
                all_class_ids.add(cls)

sorted_class_ids = sorted(all_class_ids)
class_id_map = {cid: i for i, cid in enumerate(sorted_class_ids)}

print("Class ID mapping:")
for k, v in class_id_map.items():
    print(f"  {k} -> {v}")

# STAP 2: convert per split
def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def convert_split(split):
    print(f"\n=== {split.upper()} ===")

    img_in = os.path.join(DATASET_ROOT, split, "images")
    lbl_in = os.path.join(DATASET_ROOT, split, "labels")

    img_out = os.path.join(OUTPUT_ROOT, split, "images")
    ensure_dir(img_out)

    images = []
    annotations = []
    ann_id = 1
    img_id = 1

    for img_name in sorted(os.listdir(img_in)):
        if not any(img_name.lower().endswith(e) for e in IMG_EXTS):
            continue

        src_img = os.path.join(img_in, img_name)
        dst_img = os.path.join(img_out, img_name)
        shutil.copy2(src_img, dst_img)

        with Image.open(src_img) as img:
            width, height = img.size

        images.append({
            "id": img_id,
            "file_name": img_name,
            "width": width,
            "height": height
        })

        label_path = os.path.join(
            lbl_in,
            os.path.splitext(img_name)[0] + ".txt"
        )

        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    orig_cls, xc, yc, w, h = map(float, line.split())
                    orig_cls = int(orig_cls)

                    coco_cls = class_id_map[orig_cls]

                    x = (xc - w / 2) * width
                    y = (yc - h / 2) * height
                    bw = w * width
                    bh = h * height

                    annotations.append({
                        "id": ann_id,
                        "image_id": img_id,
                        "category_id": coco_cls,
                        "bbox": [x, y, bw, bh],
                        "area": bw * bh,
                        "iscrowd": 0
                    })
                    ann_id += 1

        img_id += 1

    categories = [
        {"id": v, "name": str(k)}
        for k, v in class_id_map.items()
    ]

    coco = {
        "images": images,
        "annotations": annotations,
        "categories": categories
    }

    out_ann = os.path.join(OUTPUT_ROOT, split, "annotations.json")
    ensure_dir(os.path.dirname(out_ann))

    with open(out_ann, "w") as f:
        json.dump(coco, f, indent=2)

    print(f"Images: {len(images)}")
    print(f"Annotations: {len(annotations)}")
    print(f"Saved: {out_ann}")

# MAIN

print(f"\nInput:  {DATASET_ROOT}")
print(f"Output: {OUTPUT_ROOT}")

for split in SPLITS:
    convert_split(split)

print("\n Conversion done (YOLO → COCO with class remapping)")


Class ID mapping:
  1912 -> 0
  2351 -> 1
  2357 -> 2
  2358 -> 3
  2392 -> 4
  2501 -> 5
  2991 -> 6
  2997 -> 7
  3031 -> 8
  5121 -> 9
  5800 -> 10
  5920 -> 11
  5953 -> 12
  6315 -> 13
  6714 -> 14
  7069 -> 15
  7504 -> 16
  8083 -> 17
  8134 -> 18
  9826 -> 19
  10896 -> 20
  11304 -> 21

Input:  ../dataset/dataset_splits
Output: ../dataset/dataset_splits\../splitted_coco_dataset

=== TRAIN ===
Images: 8008
Annotations: 6597
Saved: ../dataset/dataset_splits\../splitted_coco_dataset\train\annotations.json

=== VAL ===
Images: 1001
Annotations: 826
Saved: ../dataset/dataset_splits\../splitted_coco_dataset\val\annotations.json

=== TEST ===
Images: 1002
Annotations: 816
Saved: ../dataset/dataset_splits\../splitted_coco_dataset\test\annotations.json

 Conversion done (YOLO → COCO with class remapping)
